# 🔬 Notebook 3: Netflix — Deep Dive: Encoding, Recs, CDN math

## 🛠️ Setup

```bash
cd 06-system-designs/netflix
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## Deep dive 1 — the encoding pipeline

When a new mezzanine (master) file arrives, we need to produce many renditions in parallel.

```
  upload → S3  ──▶  encoding queue (SQS/Kafka)
                        │
                  ┌─────┴──────┐
                  ▼            ▼
             ffmpeg worker   ffmpeg worker        (N workers)
                  │            │
                  ▼            ▼
            240p.ts       1080p.ts   → S3 → CDN
                  │
                  ▼
         manifest builder → catalog DB
```

Key properties:
- **Fan-out per rendition**. A 2-hour movie × 6 renditions = 6 independent jobs.
- **Chunked encoding**: split the mezzanine into, say, 3-minute segments. 40 workers per movie → done in minutes.
- **Idempotent jobs** keyed by (asset_id, rendition): retrying after a crash is safe.

Below we simulate the queue + workers with threads.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
import queue, time, random

@dataclass
class EncodeJob:
    asset_id: int
    rendition: str  # "240p" etc.

def encode(job: EncodeJob) -> str:
    # Pretend ffmpeg takes variable time
    time.sleep(random.uniform(0.05, 0.2))
    return f"{job.asset_id}-{job.rendition}.ts"

jobs_q: "queue.Queue[EncodeJob]" = queue.Queue()
for rid in ("240p","360p","480p","720p","1080p","4k"):
    jobs_q.put(EncodeJob(asset_id=42, rendition=rid))

results = []
def worker():
    while True:
        try: job = jobs_q.get_nowait()
        except queue.Empty: return
        results.append(encode(job))

t0 = time.time()
with ThreadPoolExecutor(max_workers=4) as pool:
    for _ in range(4): pool.submit(worker)
print(f"Encoded {len(results)} renditions in {time.time()-t0:.2f}s (4 parallel workers)")
for r in results: print(" ", r)


## Deep dive 2 — recommendations

Netflix's recommender is offline-precomputed so serving is just a key lookup:

```
  user_features   item_features
        └──────┬──────┘
               ▼
      ┌─────────────────┐
      │ offline training │   (nightly / weekly)
      │  collaborative   │
      │  filtering + ML  │
      └────────┬─────────┘
               │ emit top-K
               ▼
         recs_cache (Redis)
               ▲
  GET /recs/for-me → O(1) lookup
```

We'll implement a **toy item-based collaborative filter** using cosine similarity.

In [ ]:
import math
from collections import defaultdict

# user → set of liked titles (1 = liked, 0 = not watched)
ratings = {
    "alice": {"the matrix", "inception", "interstellar"},
    "bob":   {"the matrix", "inception", "memento"},
    "carol": {"interstellar", "arrival"},
    "dave":  {"memento", "inception"},
}

# Build title vectors: for each title, which users liked it
title_users = defaultdict(set)
for user, titles in ratings.items():
    for t in titles: title_users[t].add(user)

def cosine(a: set, b: set) -> float:
    if not a or not b: return 0.0
    return len(a & b) / math.sqrt(len(a) * len(b))

def recommend(user: str, k=3):
    watched = ratings[user]
    scores = defaultdict(float)
    for watched_title in watched:
        for other in title_users:
            if other in watched: continue
            scores[other] += cosine(title_users[watched_title], title_users[other])
    return sorted(scores.items(), key=lambda x: -x[1])[:k]

print("alice →", recommend("alice"))
print("carol →", recommend("carol"))


## Deep dive 3 — why CDN beats origin

Napkin math:
- 30M concurrent viewers × 3 Mbps = 90 Tbps.
- Typical data center egress cap: ~10 Tbps.
- With 200 edge POPs each serving 450 Gbps: we're in range.

CDN wins on three axes:
1. **Bandwidth**: massive aggregate capacity close to users.
2. **Latency**: 20ms RTT vs. 150ms trans-ocean.
3. **Origin protection**: one popular episode could DoS the origin; CDN absorbs the spike.

This is why Netflix built *OpenConnect* — caching boxes placed inside ISP networks.
